# Electrical Transmission Line Fault Classification

This notebook documents a supervised data mining workflow to classify electrical transmission line fault types from current and voltage measurements.

## Problem Description

Electrical fault classification helps identify fault patterns in transmission systems. In this study, the goal is to predict the fault type from measured electrical signals.

The notebook follows a reproducible workflow: dataset loading, data inspection, exploratory analysis, preprocessing, feature engineering, model training, evaluation, interpretation, and CSV export.

## Dataset Context

The dataset is downloaded from Kaggle through `kagglehub`: `esathyaprakash/electrical-fault-detection-and-classification`.

The preferred file is `classData.csv`. If this file is not found, the notebook selects the largest CSV available in the downloaded dataset directory.

## Library Imports

In [ ]:
from pathlib import Path
import os

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-cache")

import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

## Global Settings

In [ ]:
KAGGLE_DATASET = "esathyaprakash/electrical-fault-detection-and-classification"
DATASET_DIRECTORY = None
DATASET_FILE_PATH = None
TARGET_COLUMN = "fault_type"
TEST_SIZE = 0.2
RANDOM_STATE = 42
MINIMUM_RECALL_PER_CLASS = 0.90
RESULTS_PATH = Path("model_comparison_results.csv")

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("seaborn-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

## Dataset Download

In [ ]:
dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
DATASET_DIRECTORY = Path(dataset_path)

print("Dataset files path:", DATASET_DIRECTORY)
available_files = sorted(path for path in DATASET_DIRECTORY.rglob("*") if path.is_file())
available_files

In [ ]:
def select_dataset_file(dataset_directory):
    csv_files = sorted(dataset_directory.rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("No CSV file was found in the downloaded dataset directory.")

    preferred_names = ["classData.csv", "classdata.csv"]
    for preferred_name in preferred_names:
        for csv_file in csv_files:
            if csv_file.name == preferred_name:
                return csv_file

    return max(csv_files, key=lambda csv_file: csv_file.stat().st_size)


DATASET_FILE_PATH = select_dataset_file(DATASET_DIRECTORY)
print("Selected dataset file:", DATASET_FILE_PATH)

## Dataset Loading

In [ ]:
raw_data = pd.read_csv(DATASET_FILE_PATH)

print("Rows and columns:", raw_data.shape)
print("Columns:", raw_data.columns.tolist())
print("Configured target column:", TARGET_COLUMN)
display(raw_data.head())
display(raw_data.dtypes.to_frame("type"))

In [ ]:
def build_fault_type(row):
    label_columns = ["G", "C", "B", "A"]
    active_labels = [label for label in label_columns if int(row[label]) == 1]
    return "No fault" if not active_labels else "".join(active_labels)


def prepare_target(dataframe):
    label_columns = ["G", "C", "B", "A"]
    if TARGET_COLUMN in dataframe.columns:
        return dataframe.copy(), []
    if set(label_columns).issubset(dataframe.columns):
        prepared = dataframe.copy()
        prepared[TARGET_COLUMN] = prepared.apply(build_fault_type, axis=1)
        return prepared, label_columns

    candidate_columns = ["faultType", "Fault Type", "Fault_Type", "Output (S)", "Output"]
    for candidate_column in candidate_columns:
        if candidate_column in dataframe.columns:
            prepared = dataframe.rename(columns={candidate_column: TARGET_COLUMN}).copy()
            return prepared, []

    raise ValueError("The target column could not be identified. Review the dataset columns before continuing.")


data, target_source_columns = prepare_target(raw_data)
print("Target source columns:", target_source_columns or [TARGET_COLUMN])
display(data[[TARGET_COLUMN]].head())

## Initial Data Inspection

In [ ]:
print("Dataset dimensions:", data.shape)
print("Duplicate rows:", data.duplicated().sum())

display(data.info())
display(data.isna().sum().to_frame("missing_values"))
display(data.describe(include="all").T)
display(data[TARGET_COLUMN].value_counts().to_frame("count"))

## Exploratory Data Analysis

In [ ]:
analysis_feature_data = data.drop(columns=[TARGET_COLUMN])
if target_source_columns:
    analysis_feature_data = analysis_feature_data.drop(columns=target_source_columns)

numeric_columns_for_analysis = analysis_feature_data.select_dtypes(include=np.number).columns.tolist()
class_counts = data[TARGET_COLUMN].value_counts()

fig, ax = plt.subplots(figsize=(9, 4))
class_counts.plot(kind="bar", ax=ax, color="#2f6f8f")
ax.set_title("Class distribution")
ax.set_xlabel("Fault type")
ax.set_ylabel("Registros")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

class_counts

In [ ]:
data[numeric_columns_for_analysis].hist(figsize=(12, 8), bins=30, color="#3b7a57")
plt.suptitle("Numeric variable histograms", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
correlation_matrix = data[numeric_columns_for_analysis].corr()

fig, ax = plt.subplots(figsize=(8, 6))
image = ax.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_title("Correlation matrix")
ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_xticklabels(correlation_matrix.columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlation_matrix.columns)))
ax.set_yticklabels(correlation_matrix.columns)
fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

display(correlation_matrix)

In [ ]:
selected_boxplot_columns = numeric_columns_for_analysis[:6]
fig, axes = plt.subplots(len(selected_boxplot_columns), 1, figsize=(10, 3 * len(selected_boxplot_columns)))
if len(selected_boxplot_columns) == 1:
    axes = [axes]

for axis, column in zip(axes, selected_boxplot_columns):
    data.boxplot(column=column, by=TARGET_COLUMN, ax=axis, rot=45)
    axis.set_title(f"{column} by fault type")
    axis.set_xlabel("Fault type")
    axis.set_ylabel(column)

plt.suptitle("")
plt.tight_layout()
plt.show()

In [ ]:
def summarize_outliers(dataframe, columns):
    summaries = []
    for column in columns:
        first_quartile = dataframe[column].quantile(0.25)
        third_quartile = dataframe[column].quantile(0.75)
        interquartile_range = third_quartile - first_quartile
        lower_bound = first_quartile - 1.5 * interquartile_range
        upper_bound = third_quartile + 1.5 * interquartile_range
        outlier_count = ((dataframe[column] < lower_bound) | (dataframe[column] > upper_bound)).sum()
        summaries.append({"feature": column, "outliers": outlier_count, "outlier_rate": outlier_count / len(dataframe)})
    return pd.DataFrame(summaries).sort_values("outlier_rate", ascending=False)


outlier_summary = summarize_outliers(data, numeric_columns_for_analysis)
display(outlier_summary)

print("Exploratory analysis finding: class balance, target-wise dispersion, correlations, and outlier rates should be evaluated together before model interpretation.")

## Preprocessing

Duplicate rows are removed before modeling. Missing values and transformations are handled inside scikit-learn pipelines to prevent data leakage. Any transformation that learns from data is fitted only on the training data.

In [ ]:
modeling_data = data.drop_duplicates().copy()

feature_columns_to_drop = [TARGET_COLUMN] + target_source_columns
features = modeling_data.drop(columns=feature_columns_to_drop)
target = modeling_data[TARGET_COLUMN]

numeric_features = features.select_dtypes(include=np.number).columns.tolist()
categorical_features = features.select_dtypes(exclude=np.number).columns.tolist()


def build_preprocessor(training_features):
    numeric_training_features = training_features.select_dtypes(include=np.number).columns.tolist()
    categorical_training_features = training_features.select_dtypes(exclude=np.number).columns.tolist()

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    return ColumnTransformer(transformers=[
        ("numeric", numeric_transformer, numeric_training_features),
        ("categorical", categorical_transformer, categorical_training_features)
    ])


print("Rows after duplicate removal:", len(modeling_data))
print("Numeric columns:", numeric_features)
print("Categorical columns:", categorical_features)

## Feature Engineering

The new variables use physical relationships from the three-phase system to represent zero-sequence components, phase imbalance, and total current and voltage magnitude.

In [ ]:
required_three_phase_columns = ["Ia", "Ib", "Ic", "Va", "Vb", "Vc"]
missing_three_phase_columns = [column for column in required_three_phase_columns if column not in features.columns]
if missing_three_phase_columns:
    raise ValueError(f"Required columns for feature engineering were not found: {missing_three_phase_columns}")

# Physical fault classification: [0 0 0 0] = no fault, [1 0 0 1] = LG (phase A and ground), [0 0 1 1] = LL (phases A and B), [1 0 1 1] = LLG (phases A, B, and ground), [0 1 1 1] = LLL (three phases), [1 1 1 1] = LLLG (three phases and ground).
features = features.copy()
features["I_seq_zero"] = features[["Ia", "Ib", "Ic"]].mean(axis=1)
features["V_seq_zero"] = features[["Va", "Vb", "Vc"]].mean(axis=1)
features["I_unbalance"] = features[["Ia", "Ib", "Ic"]].std(axis=1)
features["V_unbalance"] = features[["Va", "Vb", "Vc"]].std(axis=1)
features["I_magnitude"] = np.sqrt((features[["Ia", "Ib", "Ic"]] ** 2).sum(axis=1))
features["V_magnitude"] = np.sqrt((features[["Va", "Vb", "Vc"]] ** 2).sum(axis=1))

numeric_features = features.select_dtypes(include=np.number).columns.tolist()
categorical_features = features.select_dtypes(exclude=np.number).columns.tolist()

engineered_features = [
    "I_seq_zero",
    "V_seq_zero",
    "I_unbalance",
    "V_unbalance",
    "I_magnitude",
    "V_magnitude",
]
print("Created features:", engineered_features)
print("Updated numeric columns:", numeric_features)

## Train and Test Split

In [ ]:
minimum_class_count = target.value_counts().min()
stratify_target = target if minimum_class_count >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=stratify_target,
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])

## Model Training

In [ ]:
models = {
    "Decision Tree": DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE),
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
}

trained_models = {}
for model_name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", build_preprocessor(X_train)),
        ("model", model),
    ])
    pipeline.fit(X_train, y_train)
    trained_models[model_name] = pipeline
    print(f"Trained model: {model_name}")

## Model Evaluation

In [ ]:
evaluation_rows = []
predictions = {}

report_column_names = {
    "precision": "precision",
    "recall": "recall",
    "f1-score": "f1-score",
    "support": "support",
}
report_index_names = {
    "accuracy": "accuracy",
    "macro avg": "macro avg",
    "weighted avg": "weighted avg",
}

for model_name, pipeline in trained_models.items():
    y_pred = pipeline.predict(X_test)
    predictions[model_name] = y_pred
    evaluation_rows.append({
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    })
    print("\n", model_name)
    report = classification_report(y_test, y_pred, zero_division=0, output_dict=True)
    report_table = pd.DataFrame(report).T.rename(columns=report_column_names).rename(index=report_index_names)
    display(report_table)

results = pd.DataFrame(evaluation_rows).sort_values(["f1_score", "recall"], ascending=False).reset_index(drop=True)
display(results.rename(columns={
    "model": "model",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1_score": "f1-score",
}))

## Per-Class Analysis

This section details each model's performance by fault class. For each trained model, the confusion matrix and a table with per-class precision, recall, and F1-score are displayed. At the end of each model analysis, alerts are printed for classes with recall below the threshold configured in `MINIMUM_RECALL_PER_CLASS`.

In [ ]:
class_labels = sorted(y_test.unique())
per_class_reports = {}

for model_name, y_pred in predictions.items():
    print(f"\nPer-class analysis - {model_name}")

    matrix = confusion_matrix(y_test, y_pred, labels=class_labels)
    fig, ax = plt.subplots(figsize=(7, 6))
    image = ax.imshow(matrix, cmap="Blues")
    ax.set_title(f"Confusion matrix - {model_name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_xticks(range(len(class_labels)))
    ax.set_xticklabels(class_labels, rotation=45, ha="right")
    ax.set_yticks(range(len(class_labels)))
    ax.set_yticklabels(class_labels)

    for row_index in range(matrix.shape[0]):
        for column_index in range(matrix.shape[1]):
            ax.text(column_index, row_index, matrix[row_index, column_index], ha="center", va="center", color="black")

    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

    report = classification_report(
        y_test,
        y_pred,
        labels=class_labels,
        output_dict=True,
        zero_division=0,
    )
    report_table = pd.DataFrame(report).T.loc[class_labels, ["precision", "recall", "f1-score", "support"]]
    report_table = report_table.rename(columns={
        "precision": "precision",
        "recall": "recall",
        "f1-score": "f1-score",
        "support": "support",
    })
    per_class_reports[model_name] = report_table
    display(report_table)

    low_recall_rows = report_table[report_table["recall"] < MINIMUM_RECALL_PER_CLASS]
    for class_name, row in low_recall_rows.iterrows():
        print(
            f"ALERT: {model_name} had recall {row['recall']:.4f} "
            f"for class {class_name}, below the configured threshold of {MINIMUM_RECALL_PER_CLASS:.2f}."
        )

## Cross-Validation

Weighted F1-score is used because it is suitable for multiclass classification and class imbalance scenarios.

In [ ]:
fold_count = min(5, int(y_train.value_counts().min()))
cross_validation_rows = []

if fold_count >= 2:
    cross_validator = StratifiedKFold(n_splits=fold_count, shuffle=True, random_state=RANDOM_STATE)
    for model_name, model in models.items():
        pipeline = Pipeline(steps=[("preprocessor", build_preprocessor(X_train)), ("model", model)])
        scores = cross_val_score(pipeline, X_train, y_train, cv=cross_validator, scoring="f1_weighted", n_jobs=1)
        cross_validation_rows.append({"model": model_name, "cv_f1_mean": scores.mean(), "cv_f1_std": scores.std()})

cross_validation_results = pd.DataFrame(cross_validation_rows)
display(cross_validation_results.rename(columns={
    "model": "model",
    "cv_f1_mean": "cross_validation_f1_mean",
    "cv_f1_std": "cross_validation_f1_std",
}))
if cross_validation_results.empty:
    print("Cross-validation was skipped because at least one class has fewer than two records.")

## Model Comparison

In [ ]:
if not cross_validation_results.empty:
    results = results.merge(cross_validation_results, on="model", how="left")

results_for_report = results.rename(columns={
    "model": "model",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1_score": "f1-score",
    "cv_f1_mean": "cross_validation_f1_mean",
    "cv_f1_std": "cross_validation_f1_std",
})
results_for_report.to_csv(RESULTS_PATH, index=False)
best_model_name = results.iloc[0]["model"]
best_model = trained_models[best_model_name]

display(results_for_report)
print("Results saved to:", RESULTS_PATH)
print("Best model:", best_model_name)

## Logistic Regression Analysis

The `model_comparison_results.csv` file shows that Logistic Regression performed much worse than tree-based models. Accuracy was approximately 0.33 and the weighted F1-score was close to 0.20, while the Decision Tree exceeded 0.88 in weighted F1-score.

This result indicates that Logistic Regression could not separate the fault classes well with a linear decision boundary. Because the classes are formed by phase and ground combinations from electrical signals, the separation tends to depend on nonlinear interactions and thresholds between current and voltage. Tree-based models capture this type of relationship more naturally.

The cross-validation mean F1-score for Logistic Regression was also low and very close to the test F1-score, so the issue does not appear to be only a poor train/test split. Accuracy close to the largest class proportion suggests behavior similar to a simple baseline, with low ability to correctly recover several classes.

In [ ]:
if not RESULTS_PATH.exists():
    print("The results file has not been generated yet. Run the Model Comparison section first.")
else:
    comparison_from_csv = pd.read_csv(RESULTS_PATH)
    logistic_row = comparison_from_csv.loc[comparison_from_csv["model"] == "Logistic Regression"].iloc[0]
    best_row = comparison_from_csv.sort_values("f1-score", ascending=False).iloc[0]
    majority_class_rate = target.value_counts(normalize=True).max()

    logistic_analysis = pd.DataFrame([
        {"indicator": "Logistic Regression accuracy", "value": logistic_row["accuracy"]},
        {"indicator": "Logistic Regression weighted F1", "value": logistic_row["f1-score"]},
        {"indicator": "Logistic Regression mean cross-validation F1", "value": logistic_row["cross_validation_f1_mean"]},
        {"indicator": "Best model weighted F1", "value": best_row["f1-score"]},
        {"indicator": "F1 difference to the best model", "value": best_row["f1-score"] - logistic_row["f1-score"]},
        {"indicator": "Largest class proportion", "value": majority_class_rate},
    ])

    display(logistic_analysis)
    print("Interpretation: Logistic Regression stayed close to the largest-class baseline in accuracy and had a low weighted F1-score, indicating difficulty correctly recovering classes in a nonlinear multiclass problem.")


## Result Visualizations

In [ ]:
metric_columns = ["accuracy", "precision", "recall", "f1_score"]
results.set_index("model")[metric_columns].plot(kind="bar", figsize=(10, 5))
plt.title("Metric comparison between models")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# The confusion matrices for all models were displayed in the Per-Class Analysis section above.
# Feature importance is displayed in the next cell.


In [ ]:
def get_feature_names(fitted_pipeline):
    fitted_preprocessor = fitted_pipeline.named_steps["preprocessor"]
    return fitted_preprocessor.get_feature_names_out()


def plot_feature_importance(model_name, top_n=12):
    pipeline = trained_models[model_name]
    model = pipeline.named_steps["model"]
    if not hasattr(model, "feature_importances_"):
        print(f"{model_name} does not expose feature_importances_.")
        return pd.DataFrame()

    importance_data = pd.DataFrame({
        "feature": get_feature_names(pipeline),
        "importance": model.feature_importances_,
    }).sort_values("importance", ascending=False)

    plot_data = importance_data.head(top_n).sort_values("importance")
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(plot_data["feature"], plot_data["importance"], color="#8a5a44")
    ax.set_title(f"Feature importance - {model_name}")
    ax.set_xlabel("Importance")
    plt.tight_layout()
    plt.show()
    return importance_data


feature_importance_by_model = {}
for model_name in ["Decision Tree", "Random Forest", "Gradient Boosting"]:
    if model_name in trained_models:
        feature_importance_by_model[model_name] = plot_feature_importance(model_name)

## Critical Interpretation

The best model is selected mainly by weighted F1-score, then by recall and confusion matrix behavior. Accuracy is used only as complementary evidence.

To keep execution suitable for an academic submission in an environment such as Google Colab, this notebook uses only direct training of the baseline models. The heavy optimization step with `GridSearchCV` was removed from this version because it significantly increased execution time and was not necessary to correctly demonstrate the data mining workflow, leakage-free preprocessing, evaluation, and interpretation.

The Decision Tree should be reviewed carefully because a single tree can overfit training patterns. Random Forest usually reduces variance by combining multiple trees and tends to be more stable. Gradient Boosting can capture nonlinear patterns sequentially. Logistic Regression works as a simple baseline and may perform poorly when class boundaries are strongly nonlinear.

Class imbalance can distort accuracy, so weighted precision, recall, and F1-score are prioritized. Confusion matrices should be inspected to identify which fault classes are most often confused. Feature importance from tree-based models helps identify which current and voltage measurements contribute most to classification.

Limitations include the use of a single public dataset, possible simulated measurements, limited feature engineering, and the absence of an external validation set. Future improvements may include validation with real operational data, model confidence calibration, and hyperparameter optimization in a separate version if more computational budget is available.

## Final Conclusion

This study examined electrical transmission line fault classification using a Kaggle dataset. The workflow downloaded and loaded the data, inspected quality issues, explored class distribution and numerical behavior, removed duplicates, created physically motivated features, created preprocessing pipelines with data leakage prevention, trained multiple classifiers, evaluated models, compared results, visualized confusion matrices and feature importance, and saved the comparison table to CSV.

The final best model should be interpreted through weighted F1-score, recall, confusion matrix errors, and feature importance. Data mining is relevant in this context because it helps transform electrical measurements into practical diagnostic support for power system operation and maintenance.